# K-Menu Voice Guide — Qwen3-TTS 1.7B LoRA — 최종본팀원 인계용 정리본. 핵심 개선사항 2가지가 반영됨:1. **리스펠링**: 발음이 부정확한 음식명 21개를 영문 알파벳으로 다시 써서(예: Tteokbokki → Duck-bok-kee) 학습 텍스트에 반영. 화면 표시용 텍스트는 원래 로마자 표기 그대로 유지.2. **instruct 파라미터**: Qwen3-TTS의 `generate_custom_voice(instruct=...)` 기능으로 GUIDE/CAUTION 톤을 구분. 재학습 없이 추론 시점에서 바로 적용됨 (7번 셀에서 직접 확인 가능).**시작 전**: `제발.zip`을 Google Drive의 `DRIVE_ZIP_PATH` 경로에 업로드해둘 것. 셀 전체가 재실행 안전하게 짜있어서(이미 된 작업은 자동 스킵), 런타임 재시작해도 처음부터 그대로 다시 돌리면 됨.

## 0. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. 환경 설정 (패키지 + 리포 클론 + 버그 패치 + 베이스 모델)모든 단계가 멱등함(이미 되어있으면 건너뜀). 새 세션이면 순서대로 전체 실행하면 됨.

In [ ]:
!apt-get -y -qq install ffmpeg sox libsox-fmt-all!pip install -q -U qwen-tts peft torchao huggingface_hub

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osDRIVE_ZIP_PATH = '/content/drive/MyDrive/k-menu-voice-guide/제발.zip'  # 실제 경로로 수정DRIVE_WORK_DIR = '/content/drive/MyDrive/k-menu-voice-guide/work'LOG_CSV = f'{DRIVE_WORK_DIR}/training_log.csv'SAMPLES_DIR = f'{DRIVE_WORK_DIR}/samples'os.makedirs(DRIVE_WORK_DIR, exist_ok=True)os.makedirs(SAMPLES_DIR, exist_ok=True)

In [ ]:
import zipfilefrom pathlib import PathRAW_ROOT = Path("/content/raw/제발")if not (RAW_ROOT / "naration_labels.csv").exists():    out_dir = Path("/content/raw")    out_dir.mkdir(parents=True, exist_ok=True)    with zipfile.ZipFile(DRIVE_ZIP_PATH) as z:        for info in z.infolist():            name = info.filename            if not (info.flag_bits & 0x800):                try:                    name = name.encode('cp437').decode('cp949')                except UnicodeDecodeError:                    pass            target = out_dir / name            if info.is_dir():                target.mkdir(parents=True, exist_ok=True)                continue            target.parent.mkdir(parents=True, exist_ok=True)            with z.open(info) as src, open(target, 'wb') as dst:                dst.write(src.read())    print("압축 해제 완료:", os.listdir("/content/raw"))else:    print("이미 해제됨")

In [ ]:
%%bashcd /contentif [ ! -d Qwen3-TTS ]; then  git clone https://github.com/QwenLM/Qwen3-TTS.git  cd Qwen3-TTS  git checkout 0c6a7cbb6c8421a46332f8c2434c7825c4c855ef  cd ..fiif [ ! -d qwen3-tts-lora-finetuning ]; then  git clone https://github.com/instavar/qwen3-tts-lora-finetuning.gitfiif [ ! -f Qwen3-TTS/finetuning/sft_12hz_lora.py ]; then  QWEN_DIR=/content/Qwen3-TTS bash qwen3-tts-lora-finetuning/scripts/apply_patches.shfiecho "리포 준비 완료"

In [ ]:
import re, os# 버그 1: tensorboard project_dir 누락sft_path = "/content/Qwen3-TTS/finetuning/sft_12hz_lora.py"with open(sft_path) as f:    content = f.read()if "project_dir=args.output_model_path" not in content:    pattern = r'(log_with="tensorboard",)(\s*\))'    new_content, n = re.subn(pattern, r'\1\n        project_dir=args.output_model_path,\2', content, count=1)    with open(sft_path, "w") as f:        f.write(new_content)    print("버그 패치 1 완료")else:    print("버그 패치 1 이미 됨")# 버그 2: main() 호출 누락infer_path = "/content/Qwen3-TTS/finetuning/infer_lora_custom_voice.py"with open(infer_path) as f:    infer_content = f.read()if not infer_content.rstrip().endswith("main()"):    with open(infer_path, "a") as f:        f.write("    main()\n")    print("버그 패치 2 완료")else:    print("버그 패치 2 이미 됨")# 버그 3: --instruct 인자 추가 (톤 조절 핵심 기능)if "--instruct" not in infer_content:    old1 = '    parser.add_argument("--language", default="auto")\n'    new1 = old1 + '    parser.add_argument("--instruct", default="")\n'    infer_content2 = infer_content.replace(old1, new1)    old2 = "        language=args.language,\n"    new2 = old2 + "        instruct=args.instruct,\n"    infer_content2 = infer_content2.replace(old2, new2)    with open(infer_path, "w") as f:        f.write(infer_content2)    print("--instruct 인자 패치 완료")else:    print("--instruct 이미 패치됨")# 버그 4: 셸 스크립트에도 INSTRUCT 환경변수 추가infer_sh_path = "/content/qwen3-tts-lora-finetuning/scripts/run_lora_infer.sh"with open(infer_sh_path) as f:    sh_content = f.read()if "INSTRUCT" not in sh_content:    sh_content = sh_content.replace(        'MAX_NEW_TOKENS="${MAX_NEW_TOKENS:-4096}"',        'MAX_NEW_TOKENS="${MAX_NEW_TOKENS:-4096}"\nINSTRUCT="${INSTRUCT:-}"'    )    sh_content = sh_content.replace(        '  --output_wav "${OUT_WAV}"',        '  --output_wav "${OUT_WAV}" \\\n  --instruct "${INSTRUCT}"'    )    with open(infer_sh_path, "w") as f:        f.write(sh_content)    print("셸 스크립트 INSTRUCT 패치 완료")else:    print("셸 스크립트 INSTRUCT 이미 패치됨")

In [ ]:
from huggingface_hub import snapshot_downloadBASE_MODEL_LOCAL = "/content/Qwen/Qwen3-TTS-12Hz-1.7B-Base"if not os.path.exists(f"{BASE_MODEL_LOCAL}/config.json"):    snapshot_download(repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base", local_dir=BASE_MODEL_LOCAL)    print("모델 다운로드 완료")else:    print("모델 이미 있음")

## 2. 학습 데이터 빌드 (리스펠링 21개 반영)청취 테스트로 확정한 `RESPELL_MAP`을 텍스트에 자동 적용함. 오디오 파일은 그대로, 텍스트 라벨만 치환됨.

In [ ]:
%%writefile /content/build_qwen_tts_dataset.pyimport csv, json, random, re, subprocessfrom pathlib import PathRAW_ROOT = Path("/content/raw/제발")OUT_WAV_DIR = Path("/content/data/wav")OUT_JSONL_DIR = Path("/content/data")SEED = 42VAL_RATIO = 0.1OVERSAMPLE_CAUTION = 3TEXT_FIXES = {    "케리어/케리어 80.m4a": "A traditional spicy beef soup Yukgaejang is widely enjoyed.",    "케리어/케리어 215.m4a": "For a light seafood meal choose Mulhoe.",}SUSPECT_PATTERN = re.compile(r"(\.\s?s|\s[a-zA-Z])\s*$")RESPELL_MAP = {    "Tteokbokki": "Duck-bok-kee",    "Bibimbap": "Bee-bim-bahp",    "Nakji Bokkeum": "Nahk-jee Boh-kum",    "Agujjim": "Ah-goo-jim",    "Maeuntang": "May-oon-tang",    "Bibim Naengmyeon": "Bee-bim Nang-myun",    "Cheonggukjang": "Chung-gook-jang",    "Doenjang Jjigae": "Dwen-jang Jee-geh",    "Dongtae Jjigae": "Dong-teh Jee-geh",    "Dubu Kimchi": "Doo-boo Kim-chee",    "Jeyuk Bokkeum": "Jeh-yook Bok-kum",    "Kimchi Bokkeumbap": "Kim-chee Bok-kum-bap",    "Kimchi Jjigae": "Kim-chee Jee-geh",    "Kongguksu": "Kong-gook-soo",    "Mulhoe": "Mul-hweh",    "Ojingeo Bokkeum": "Oh-jing-uh Bok-kum",    "Sundae": "Soon-deh",    "Sundubu Jjigae": "Soon-doo-boo Jee-geh",    "Tteokgalbi": "Duck-gal-bee",    "Tteokguk": "Duck-gook",    "Yukhoe": "Yook-hweh",}_SORTED_NAMES = sorted(RESPELL_MAP, key=len, reverse=True)def apply_respelling(text):    for name in _SORTED_NAMES:        text = re.sub(rf"\b{re.escape(name)}\b", RESPELL_MAP[name], text)    return textdef convert_to_wav(src, dst):    dst.parent.mkdir(parents=True, exist_ok=True)    if dst.exists():        return True    result = subprocess.run(        ["ffmpeg", "-y", "-i", str(src), "-ar", "24000", "-ac", "1", "-sample_fmt", "s16", str(dst)],        capture_output=True, text=True    )    if result.returncode != 0:        print(f"[FFMPEG FAIL] {src}")        print(result.stderr[-300:])        return False    return Truedef load_csv(path):    with open(path, encoding="utf-8-sig") as f:        return [r for r in csv.DictReader(f) if r.get("audio_path")]def extract_index(audio_path):    m = re.search(r"(\d+)\.m4a$", audio_path)    return int(m.group(1))def main():    random.seed(SEED)    narration_rows = load_csv(RAW_ROOT / "naration_labels.csv")    carry_rows = load_csv(RAW_ROOT / "carry_labels.csv")    style_by_food = {}    with open(RAW_ROOT / "naration.txt", encoding="utf-8-sig") as f:        reader = csv.reader(f)        next(reader)        for parts in reader:            if not parts:                continue            _id, food_id, style = parts[0], parts[1], parts[2]            style_by_food[int(food_id)] = style.strip().upper()    samples = []    for row in narration_rows:        idx = extract_index(row["audio_path"])        text = TEXT_FIXES.get(row["audio_path"], row["text"]).strip()        text = apply_respelling(text)        samples.append({"audio_path": row["audio_path"], "text": text, "food_id": idx, "kind": "narration"})    for row in carry_rows:        idx = extract_index(row["audio_path"])        food_id = (idx - 1) // 5 + 1        text = TEXT_FIXES.get(row["audio_path"], row["text"]).strip()        text = apply_respelling(text)        samples.append({"audio_path": row["audio_path"], "text": text, "food_id": food_id, "kind": "carry"})    for s in samples:        s["style"] = style_by_food.get(s["food_id"], "GUIDE")    n_caution = sum(1 for s in samples if s["style"] == "CAUTION")    print(f"style balance -> GUIDE: {len(samples)-n_caution}  CAUTION: {n_caution}  ({n_caution/len(samples):.1%})")    ok_samples, failed = [], []    for s in samples:        src = RAW_ROOT / s["audio_path"]        dst = OUT_WAV_DIR / Path(s["audio_path"]).with_suffix(".wav")        if convert_to_wav(src, dst):            s["wav_path"] = str(dst)            ok_samples.append(s)        else:            failed.append(s["audio_path"])    print(f"변환 성공: {len(ok_samples)} / 실패: {len(failed)}")    if failed:        print("실패 목록:", failed)    samples = ok_samples    guide_c = [s for s in samples if s["kind"] == "narration" and s["style"] == "GUIDE"]    ref_guide = guide_c[0]["wav_path"] if guide_c else samples[0]["wav_path"]    for s in samples:        s["ref_audio"] = ref_guide    print(f"ref_audio (전 샘플 공통) = {ref_guide}")    expanded = []    for s in samples:        reps = OVERSAMPLE_CAUTION if s["style"] == "CAUTION" else 1        expanded.extend([s] * reps)    samples = expanded    by_group = {}    for s in samples:        by_group.setdefault((s["food_id"], s["style"]), []).append(s)    train, val = [], []    for group, items in by_group.items():        random.shuffle(items)        n_val = max(1, round(len(items) * VAL_RATIO)) if len(items) > 1 else 0        val.extend(items[:n_val])        train.extend(items[n_val:])    OUT_JSONL_DIR.mkdir(parents=True, exist_ok=True)    for name, rows in [("train_raw.jsonl", train), ("val_raw.jsonl", val)]:        with open(OUT_JSONL_DIR / name, "w", encoding="utf-8") as f:            for s in rows:                f.write(json.dumps({"audio": s["wav_path"], "text": s["text"], "ref_audio": s["ref_audio"]}, ensure_ascii=False) + "\n")        print(f"wrote {len(rows)} rows -> {OUT_JSONL_DIR / name}")if __name__ == "__main__":    main()

In [ ]:
!python /content/build_qwen_tts_dataset.py

## 3. 오디오 코드 추출 + Drive 백업

In [ ]:
%%bashcd /contentpython Qwen3-TTS/finetuning/prepare_data.py --device cuda:0 --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz --input_jsonl data/train_raw.jsonl --output_jsonl data/train_with_codes.jsonlpython Qwen3-TTS/finetuning/prepare_data.py --device cuda:0 --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz --input_jsonl data/val_raw.jsonl --output_jsonl data/val_with_codes.jsonl

In [ ]:
!mkdir -p "$DRIVE_WORK_DIR/data"!cp /content/data/*.jsonl "$DRIVE_WORK_DIR/data/"print("백업 완료")

## 4. HF 로그인Colab 왼쪽 패널에서 키(🔑) 아이콘 → `HF_TOKEN` 시크릿 등록(write 토큰) + 노트북 액세스 켜져있어야 함.

In [ ]:
from google.colab import userdataHF_TOKEN = userdata.get('HF_TOKEN')from huggingface_hub import whoamiinfo = whoami(token=HF_TOKEN)print(info["name"], info.get("auth", {}).get("accessToken", {}).get("role"))

## 5. 학습 + 자동 로깅 + 미리듣기 (**instruct 반영됨**)`run_training()`은 학습이 끝나면 체크포인트로 GUIDE/CAUTION 샘플을 자동 생성하는데, 이때 **각 스타일마다 다른 `instruct`(톤 지시 문구)를 같이 넘기는 게 핵심**임. 재학습 없이도 동작함.

In [ ]:
import subprocess, re, csv, os, timefrom IPython.display import Audio, displayGUIDE_TEXT = "Bibimbap is a colorful bowl of warm rice topped with seasoned vegetables, meat, and a fried egg."CAUTION_TEXT = "Careful, this one's spicy! Tteokbokki is chewy rice cakes simmered in a thick, spicy chili sauce."# 핵심: 스타일별 instruct 문구. 청취 테스트로 확정됨.GUIDE_INSTRUCT = "Speak in a calm, informative tone, like a knowledgeable guide explaining something interesting. Keep a natural, steady pace, not slow."CAUTION_INSTRUCT = "Say it in an urgent, warning tone, as if cautioning someone."CSV_FIELDS = ["run_id","epoch","val_loss","lr","epochs_total","lora_rank","lora_alpha",              "batch_size","mixed_precision","duration_sec","returncode",              "guide_sample_path","caution_sample_path","notes","timestamp"]def _append_csv_rows(rows_data):    file_exists = os.path.exists(LOG_CSV)    with open(LOG_CSV, "a", newline="") as f:        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)        if not file_exists:            writer.writeheader()        for row in rows_data:            writer.writerow(row)def run_training(run_id, epochs, lr, lora_rank=16, lora_alpha=32, batch_size=4,                  mixed_precision="bf16", lora_scale=0.3, notes=""):    output_dir = f"/content/output_{run_id}"    cmd = f'''cd /content && \QWEN_DIR=/content/Qwen3-TTS \TRAIN_JSONL=/content/data/train_with_codes.jsonl \VAL_JSONL=/content/data/val_with_codes.jsonl \OUTPUT_DIR={output_dir} \LR={lr} \EPOCHS={epochs} \LORA_RANK={lora_rank} \LORA_ALPHA={lora_alpha} \BATCH_SIZE={batch_size} \MIXED_PRECISION={mixed_precision} \ATTN_IMPL=sdpa \bash qwen3-tts-lora-finetuning/scripts/run_lora_train.sh'''    epoch_pattern = re.compile(r"Epoch (\d+) \| Val Loss: ([\d.]+)")    rows = []    start = time.time()    proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE,                             stderr=subprocess.STDOUT, text=True, bufsize=1)    for line in proc.stdout:        print(line, end="")        m = epoch_pattern.search(line)        if m:            rows.append((int(m.group(1)), float(m.group(2))))    proc.wait()    duration = time.time() - start    guide_path = caution_path = ""    if rows and proc.returncode == 0:        last_epoch = rows[-1][0]        ckpt = f"{output_dir}/checkpoint-epoch-{last_epoch}"        guide_path = f"{SAMPLES_DIR}/{run_id}_guide.wav"        caution_path = f"{SAMPLES_DIR}/{run_id}_caution.wav"        jobs = [(GUIDE_TEXT, GUIDE_INSTRUCT, guide_path), (CAUTION_TEXT, CAUTION_INSTRUCT, caution_path)]        for text, instruct, out_path in jobs:            infer_cmd = f'''cd /content && \QWEN_DIR=/content/Qwen3-TTS \BASE_MODEL={BASE_MODEL_LOCAL} \ADAPTER_DIR={ckpt} \LORA_SCALE={lora_scale} \ATTN_IMPL=sdpa \TEXT="{text}" \INSTRUCT="{instruct}" \OUT_WAV={out_path} \bash qwen3-tts-lora-finetuning/scripts/run_lora_infer.sh'''            r = subprocess.run(["bash", "-c", infer_cmd], capture_output=True, text=True)            if r.returncode != 0:                print(f"[미리듣기 생성 실패: {out_path}]")                print(r.stderr[-500:])    csv_rows = [{        "run_id": run_id, "epoch": epoch, "val_loss": val_loss, "lr": lr,        "epochs_total": epochs, "lora_rank": lora_rank, "lora_alpha": lora_alpha,        "batch_size": batch_size, "mixed_precision": mixed_precision,        "duration_sec": round(duration, 1), "returncode": proc.returncode,        "guide_sample_path": guide_path, "caution_sample_path": caution_path,        "notes": notes, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),    } for epoch, val_loss in rows]    _append_csv_rows(csv_rows)    final = rows[-1][1] if rows else "N/A"    print(f"\n[{run_id}] 끝 (rc={proc.returncode}, {duration:.1f}초, final val_loss={final})")    if guide_path and os.path.exists(guide_path):        print("GUIDE 미리듣기 (instruct 반영):")        display(Audio(guide_path))    if caution_path and os.path.exists(caution_path):        print("CAUTION 미리듣기 (instruct 반영):")        display(Audio(caution_path))    return proc.returncode, output_dir, rowsdef list_runs():    import pandas as pd    if not os.path.exists(LOG_CSV):        print("기록 없음")        return None    df = pd.read_csv(LOG_CSV)    return df.groupby("run_id").agg(        min_val_loss=("val_loss","min"), final_val_loss=("val_loss","last"),        epochs_total=("epochs_total","last"), lr=("lr","last"),        lora_rank=("lora_rank","last"), notes=("notes","last"),        timestamp=("timestamp","last"),    ).reset_index()def play_run(run_id):    import pandas as pd    df = pd.read_csv(LOG_CSV)    sub = df[df["run_id"] == run_id]    if sub.empty:        print(f"run_id '{run_id}' 없음")        return    print(sub[["epoch","val_loss"]].to_string(index=False))    best = sub.loc[sub["val_loss"].idxmin()]    print(f"\n최저 Val Loss: epoch {int(best['epoch'])} ({best['val_loss']:.4f})")    guide_path = sub.iloc[-1]["guide_sample_path"]    caution_path = sub.iloc[-1]["caution_sample_path"]    if isinstance(guide_path, str) and os.path.exists(guide_path):        print("GUIDE 샘플:"); display(Audio(guide_path))    if isinstance(caution_path, str) and os.path.exists(caution_path):        print("CAUTION 샘플:"); display(Audio(caution_path))print("함수 로드 완료")

## 6. 최종 학습 실행 (확정된 최적 하이퍼파라미터)

In [ ]:
rc, out_dir, rows = run_training(    run_id="final_respelled_lr5e-6_ep40",    epochs=40,    lr="5e-6",    notes="리스펠링 21개 + instruct 반영된 최종 학습",)

In [ ]:
!mkdir -p "$DRIVE_WORK_DIR/output"!cp -r {out_dir} "$DRIVE_WORK_DIR/output/"print("백업 완료")

## 7. instruct 톤 비교 데모 (효과 직접 확인)같은 문장을 GUIDE 지시 vs CAUTION 지시로 각각 생성해서 톤 차이를 직접 들어보는 셀.

In [ ]:
CKPT = f"{out_dir}/checkpoint-epoch-39"  # 실제 에폭으로 수정demo_text = "Tteokbokki is chewy rice cakes simmered in a thick, spicy chili sauce."for label, instruct in [("GUIDE로 읽기", GUIDE_INSTRUCT), ("CAUTION으로 읽기", CAUTION_INSTRUCT)]:    out_wav = f"/content/instruct_demo_{label}.wav"    cmd = f'''cd /content && \QWEN_DIR=/content/Qwen3-TTS \BASE_MODEL={BASE_MODEL_LOCAL} \ADAPTER_DIR={CKPT} \LORA_SCALE=0.3 \ATTN_IMPL=sdpa \TEXT="{demo_text}" \INSTRUCT="{instruct}" \OUT_WAV={out_wav} \bash qwen3-tts-lora-finetuning/scripts/run_lora_infer.sh'''    r = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)    print(label, "OK" if r.returncode == 0 else r.stderr[-300:])    if r.returncode == 0:        display(Audio(out_wav))

## 8. 프로덕션 통합 함수 (리스펠링 + instruct 자동 처리)팀 서비스 코드에 그대로 가져다 쓸 수 있는 함수. `narration_text`(화면 표시용 원문)와 `style`(GUIDE/CAUTION)만 넘기면 됨.

In [ ]:
INSTRUCT_MAP = {"GUIDE": GUIDE_INSTRUCT, "CAUTION": CAUTION_INSTRUCT}def generate_narration_audio(narration_text, style, output_wav="output.wav", ckpt=None, lora_scale=0.3):    ckpt = ckpt or CKPT    style = style.upper()    tts_text = apply_respelling(narration_text)    instruct = INSTRUCT_MAP[style]    cmd = f'''cd /content && \QWEN_DIR=/content/Qwen3-TTS \BASE_MODEL={BASE_MODEL_LOCAL} \ADAPTER_DIR={ckpt} \LORA_SCALE={lora_scale} \ATTN_IMPL=sdpa \TEXT="{tts_text}" \INSTRUCT="{instruct}" \OUT_WAV={output_wav} \bash qwen3-tts-lora-finetuning/scripts/run_lora_infer.sh'''    r = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)    if r.returncode != 0:        raise RuntimeError(r.stderr[-1000:])    return output_wav# apply_respelling: 2번 셀의 build_qwen_tts_dataset.py와 동일한 RESPELL_MAP 사용import re as _re_RESPELL_MAP = {    "Tteokbokki": "Duck-bok-kee", "Bibimbap": "Bee-bim-bahp",    "Nakji Bokkeum": "Nahk-jee Boh-kum", "Agujjim": "Ah-goo-jim", "Maeuntang": "May-oon-tang",    "Bibim Naengmyeon": "Bee-bim Nang-myun", "Cheonggukjang": "Chung-gook-jang",    "Doenjang Jjigae": "Dwen-jang Jee-geh", "Dongtae Jjigae": "Dong-teh Jee-geh",    "Dubu Kimchi": "Doo-boo Kim-chee", "Jeyuk Bokkeum": "Jeh-yook Bok-kum",    "Kimchi Bokkeumbap": "Kim-chee Bok-kum-bap", "Kimchi Jjigae": "Kim-chee Jee-geh",    "Kongguksu": "Kong-gook-soo", "Mulhoe": "Mul-hweh", "Ojingeo Bokkeum": "Oh-jing-uh Bok-kum",    "Sundae": "Soon-deh", "Sundubu Jjigae": "Soon-doo-boo Jee-geh",    "Tteokgalbi": "Duck-gal-bee", "Tteokguk": "Duck-gook", "Yukhoe": "Yook-hweh",}_SORTED = sorted(_RESPELL_MAP, key=len, reverse=True)def apply_respelling(text):    for name in _SORTED:        text = _re.sub(rf"\b{_re.escape(name)}\b", _RESPELL_MAP[name], text)    return text# 사용 예시generate_narration_audio(    narration_text="Careful, this one's spicy! Tteokbokki is chewy rice cakes simmered in a thick, spicy chili sauce.",    style="CAUTION",    output_wav="/content/demo_final.wav",)display(Audio("/content/demo_final.wav"))

## 9. 허깅페이스 업로드

In [ ]:
REPO_ID = "k-bite-team/k-menu-qwen3-tts-lora"  # 조직 권한 미해결 시 minggyeong/... 으로 변경from huggingface_hub import HfApiapi = HfApi(token=HF_TOKEN)api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True, private=False, token=HF_TOKEN)api.upload_folder(folder_path=CKPT, repo_id=REPO_ID, repo_type="model", token=HF_TOKEN,                   commit_message="Final adapter: respelling + instruct tone control")print(f"업로드 완료: https://huggingface.co/{REPO_ID}")